# 각인(imprint) 인식 모델 - Colab 학습

**이 노트북을 실행하기 전에 로컬/서버에서 먼저 해두셔야 하는 것:**

1. 로컬 PC에서 압축/준비:
   ```powershell
   tar -cf crops.tar -C ML\IMPRINT\data crops
   # labels.csv는 그대로(ML\IMPRINT\data\labels.csv)
   ```
2. AWS 서버로 업로드:
   ```bash
   scp crops.tar labels.csv ML\IMPRINT\train_imprint_model.py bomul@54.180.81.50:~/git/boyak-be/ML/IMPRINT/data/
   ```
3. 서버 nginx에 임시 static 경로 추가 (기존 FastAPI 서비스는 안 건드림 - 443/HTTPS 그대로 재사용):
   ```bash
   sudo vi /etc/nginx/sites-available/boyak-be.conf
   ```
   443 서버 블록 안에 추가:
   ```nginx
   location /imprint_data/ {
       alias /home/bomul/git/boyak-be/ML/IMPRINT/data/;
   }
   ```
   ```bash
   sudo nginx -t && sudo systemctl reload nginx
   ```

다 되셨으면 아래 순서대로 실행하세요. **런타임 → 런타임 유형 변경 → GPU(T4)** 먼저 선택해두는 것도 잊지 마세요.

**다 받은 뒤엔 꼭 이 location 블록을 지우고 다시 `nginx -t && systemctl reload nginx`로 원복하세요.**

## 0. GPU 확인

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 안 잡혔습니다 - 런타임 유형을 GPU로 바꾸고 다시 연결하세요.")

nginx의 임시 static 경로(/imprint_data/)를 통해 HTTPS로 3개 파일(crops.tar, labels.csv, train_imprint_model.py)을 받습니다. 별도 포트/보안그룹 필요 없습니다.

In [ ]:
SERVER_BASE = "https://api.boyak.bomul.ai.kr/imprint_data"

!wget -q --show-progress "$SERVER_BASE/crops.tar" -O crops.tar
!wget -q --show-progress "$SERVER_BASE/labels.csv" -O labels.csv
!wget -q --show-progress "$SERVER_BASE/train_imprint_model.py" -O train_imprint_model.py
!ls -lh crops.tar labels.csv train_imprint_model.py

## 2. 압축 해제 + 배치

In [ ]:
!mkdir -p /content/imprint_ml/data
!tar -xf crops.tar -C /content/imprint_ml/data   # crops/ 폴더로 풀림
!cp labels.csv /content/imprint_ml/data/labels.csv
!cp train_imprint_model.py /content/imprint_ml/train_imprint_model.py

%cd /content/imprint_ml
!ls -la
!echo "---"
!ls data
!echo "---"
!wc -l data/labels.csv
!ls data/crops | wc -l

## 3. 의존성 확인

Colab엔 보통 torch/numpy/opencv가 기본 설치돼 있습니다. 혹시 `cv2` import 에러가 나면 아래 셀의 주석을 풀고 실행하세요.

In [ ]:
# !pip install -q opencv-python-headless

import cv2
import numpy as np
import torch
print("cv2", cv2.__version__, "/ torch", torch.__version__)

## 4. 학습 실행

GPU 메모리 여유가 있으니 `--batch-size`는 로컬(CPU)보다 올려서 씁니다. 필요하면 `--epochs`, `--lr`도 조정하세요.

In [ ]:
!python train_imprint_model.py --epochs 50 --batch-size 64

## 5. 학습된 모델 다운로드

`ML/IMPRINT/models/imprint_crnn.pt`로 저장됩니다(에폭마다 검증 정확도가 갱신될 때만 저장). 로컬로 받아가세요.

In [ ]:
from google.colab import files
files.download("models/imprint_crnn.pt")

## 6. (중요) 다 받았으면 서버 정리

이 노트북에서 실행하는 게 아니라 **서버(AWS)에 직접 접속해서** 하셔야 합니다:

1. `/etc/nginx/sites-available/boyak-be.conf`에서 `location /imprint_data/ { ... }` 블록 삭제
2. `sudo nginx -t && sudo systemctl reload nginx`로 원복
3. `rm -f /home/bomul/git/boyak-be/ML/IMPRINT/data/crops.tar` (업로드했던 tar는 이제 필요 없으니 정리)

FastAPI 서비스는 건드린 적 없어서 재시작 불필요합니다.